# What held-out diagnostics can establish

A lag-one correlation for feature $j$ compares adjacent values across the held-out windows:

$$
\widehat\rho_j=
\frac{\sum_k(x_{k,j}-\bar x_j)(y_{k,j}-\bar y_j)}
{\left[\sum_k(x_{k,j}-\bar x_j)^2\sum_k(y_{k,j}-\bar y_j)^2\right]^{1/2}}.
$$

Here, $x_{k,j}$ is a feature value at one position; $y_{k,j}$ is the next value in the same window; $k$ indexes all adjacent pairs; $\bar x_j$ and $\bar y_j$ are their means. The absolute difference between real and generated correlations measures one narrow aspect of temporal dependence.

For next-step prediction, the squared error is

$$
\operatorname{MSE}=\frac{1}{Nd}\sum_{i=1}^{N}\sum_{j=1}^{d}(Y_{i,j}-\widehat Y_{i,j})^2.
$$

Here, $N$ is the number of held-out windows; $d$ is the feature count; $Y_{i,j}$ is the last value of feature $j$ in held-out window $i$; $\widehat Y_{i,j}$ is its prediction from the preceding values. One predictor is fitted to generated windows and another to observed training windows.

The classifier for observed and generated windows is summarized by balanced accuracy:

$$
\operatorname{BA}=\frac{1}{2}\left(\frac{\mathrm{TP}}{\mathrm{TP}+\mathrm{FN}}+\frac{\mathrm{TN}}{\mathrm{TN}+\mathrm{FP}}\right).
$$

Here, $\mathrm{TP}$ and $\mathrm{TN}$ are correctly classified real and generated windows; $\mathrm{FN}$ and $\mathrm{FP}$ are the corresponding errors. A value near one half indicates weak separation by this particular classifier, not proof that the two distributions are equal.

## Fresh Generated Windows

In [1]:
from pathlib import Path
import json
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display

def display(value):
    module = value.__class__.__module__
    if module.startswith('pandas') and hasattr(value, '_repr_html_'):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + '</div>'))
    return _display(value)
BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
import RiskLabAI
assert RiskLabAI.__version__ == '3.2.0'
DATA_PATH = ensure_dataset(BOOK_ROOT, 'fred_nasdaq')
FIGURES = BOOK_ROOT / 'outputs' / 'figures' / '03_generative_financial_time_series'
TABLES = BOOK_ROOT / 'outputs' / 'tables' / '03_generative_financial_time_series'
FIGURE_DATA = BOOK_ROOT / 'outputs' / 'figure_data' / '03_generative_financial_time_series'
ARTIFACTS = BOOK_ROOT / 'outputs' / 'artifacts' / '03_generative_financial_time_series'
for directory in [FIGURES, TABLES, FIGURE_DATA, ARTIFACTS]:
    directory.mkdir(parents=True, exist_ok=True)
PALETTE = {'background': '#111827', 'axes': '#111827', 'text': '#E5E7EB', 'cyan': '#38BDF8', 'amber': '#FBBF24', 'green': '#34D399', 'rose': '#FB7185', 'violet': '#C084FC', 'grid': '#374151'}
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11.5, 'figure.facecolor': PALETTE['background'], 'axes.facecolor': PALETTE['axes'], 'axes.edgecolor': PALETTE['text'], 'axes.labelcolor': PALETTE['text'], 'axes.titlecolor': PALETTE['text'], 'xtick.color': PALETTE['text'], 'ytick.color': PALETTE['text'], 'text.color': PALETTE['text'], 'grid.color': PALETTE['grid'], 'grid.linestyle': ':', 'legend.facecolor': PALETTE['axes'], 'legend.edgecolor': PALETTE['grid'], 'savefig.facecolor': PALETTE['background'], 'savefig.transparent': False, 'savefig.dpi': 300})

def rendered_value(value):
    if pd.isna(value):
        return ''
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f'{int(value):,}'
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if number == 0:
            return '0'
        if number.is_integer():
            return f'{int(number):,}'
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f'{number:.6g}'
        return f'{number:.6f}'.rstrip('0').rstrip('.')
    return str(value)

def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))

def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f'{stem}.csv', index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position='htbp')
    if fit:
        latex = latex.replace('\\begin{tabular}', '\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}', 1).replace('\\end{tabular}', '\\end{tabular}\\end{adjustbox}', 1)
    (TABLES / f'{stem}.tex').write_text(latex, encoding='utf-8')
    display(shown)
    return shown

def finish_axis(axis, grid=True):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)

def save_figure(figure, stem):
    figure.savefig(FIGURES / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    figure.savefig(FIGURES / f'{stem}.pdf', bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
import torch
from RiskLabAI.data.synthetic_data.timegan import TimeGAN, timegan_quality_report
payload = np.load(ARTIFACTS / 'prepared_timegan_data.npz')
training_scaled = payload['training']
validation_scaled = payload['validation']
minimum = payload['minimum']
scale = payload['scale']
feature_names = payload['feature_names'].tolist()
window_length = int(payload['window_length'][0])
model = TimeGAN(features=2, hidden_size=8, seed=4104)
state = torch.load(ARTIFACTS / 'timegan_state.pt', map_location='cpu', weights_only=True)
model.load_state_dict(state)
real_training = training_scaled * scale + minimum
real_validation = validation_scaled * scale + minimum
synthetic_training_scaled = model.sample(len(real_training), window_length, seed=4106).numpy()
synthetic_validation_scaled = model.sample(len(real_validation), window_length, seed=4107).numpy()
synthetic_training = synthetic_training_scaled * scale + minimum
synthetic_validation = synthetic_validation_scaled * scale + minimum
assert synthetic_training.shape == real_training.shape
assert synthetic_validation.shape == real_validation.shape
assert np.isfinite(synthetic_training).all()
assert np.isfinite(synthetic_validation).all()


In [2]:
quality = timegan_quality_report(real_training, real_validation, synthetic_training, synthetic_validation)

random_generator = np.random.default_rng(4110)
shuffled_training = real_training.copy()
shuffled_validation = real_validation.copy()
for index in range(len(shuffled_training)):
    shuffled_training[index] = shuffled_training[index, random_generator.permutation(window_length)]
for index in range(len(shuffled_validation)):
    shuffled_validation[index] = shuffled_validation[index, random_generator.permutation(window_length)]
shuffled_quality = timegan_quality_report(real_training, real_validation, shuffled_training, shuffled_validation)

lag_table = pd.DataFrame({
    "Feature": feature_names,
    "Observed held-out correlation": quality["real_lag_one_correlation"],
    "Generated correlation": quality["synthetic_lag_one_correlation"],
    "Generated absolute gap": quality["lag_one_absolute_gap"],
    "Time-shuffled correlation": shuffled_quality["synthetic_lag_one_correlation"],
    "Time-shuffled absolute gap": shuffled_quality["lag_one_absolute_gap"],
})
lag_table.to_csv(FIGURE_DATA / "lag_correlation_comparison.csv", index=False)
save_table(lag_table, "lag_correlation_diagnostics", "Lag-one correlations and absolute gaps on the held-out period.", "tab:chapter4-lag-diagnostics", fit=True)

scalar_quality = pd.DataFrame({
    "Diagnostic": ["Next-step error after fitting generated windows", "Next-step error after fitting observed windows", "Balanced accuracy for observed and generated windows"],
    "Generated windows": [quality["synthetic_trained_prediction_mse"], quality["real_trained_prediction_mse"], quality["balanced_discriminator_accuracy"]],
    "Time-shuffled windows": [shuffled_quality["synthetic_trained_prediction_mse"], shuffled_quality["real_trained_prediction_mse"], shuffled_quality["balanced_discriminator_accuracy"]],
})
scalar_quality.to_csv(FIGURE_DATA / "scalar_quality_comparison.csv", index=False)
save_table(scalar_quality, "held_out_quality", "Held-out predictive and discrimination diagnostics.", "tab:chapter4-held-out-quality", fit=True)

,Feature,Observed held-out correlation,Generated correlation,Generated absolute gap,Time-shuffled correlation,Time-shuffled absolute gap
0,Daily log return (%),-0.145809,0.533994,0.679803,-0.007594,0.138215
1,Twenty-day return volatility (%),0.990743,0.996314,0.005571,0.862719,0.128024


,Diagnostic,Generated windows,Time-shuffled windows
0,Next-step error after fitting generated windows,3.156237,1.338769
1,Next-step error after fitting observed windows,1.262549,1.262549
2,Balanced accuracy for observed and generated w...,0.964736,0.529282


,Diagnostic,Generated windows,Time-shuffled windows
0,Next-step error after fitting generated windows,3.156237,1.338769
1,Next-step error after fitting observed windows,1.262549,1.262549
2,Balanced accuracy for observed and generated w...,0.964736,0.529282


In [3]:
moment_rows = []
distribution_rows = []
sources = [("Observed held-out", real_validation), ("Generated", synthetic_validation), ("Time-shuffled", shuffled_validation)]
for source, windows in sources:
    for feature_index, feature in enumerate(feature_names):
        values = windows[:, :, feature_index].reshape(-1)
        moment_rows.append({
            "Source": source,
            "Feature": feature,
            "Mean": values.mean(),
            "Standard deviation": values.std(ddof=1),
            "5th percentile": np.quantile(values, 0.05),
            "Median": np.median(values),
            "95th percentile": np.quantile(values, 0.95),
        })
        selected = values[np.linspace(0, len(values) - 1, 12000, dtype=int)] if len(values) > 12000 else values
        distribution_rows.extend({"Source": source, "Feature": feature, "Value": value} for value in selected)
moment_table = pd.DataFrame(moment_rows)
distribution_values = pd.DataFrame(distribution_rows)
distribution_values.to_csv(FIGURE_DATA / "distribution_values.csv", index=False)
save_table(moment_table, "distribution_summary", "Held-out feature summaries for observed, generated, and time-shuffled windows.", "tab:chapter4-distribution-summary", fit=True)

selected_indices = [0, 400, 800, 1200]
selected_rows = []
for source, windows in [("Observed held-out", real_validation), ("Generated", synthetic_validation)]:
    for window_number, index in enumerate(selected_indices, start=1):
        for day in range(window_length):
            for feature_index, feature in enumerate(feature_names):
                selected_rows.append({"Source": source, "Window": window_number, "Relative day": day + 1, "Feature": feature, "Value": windows[index, day, feature_index]})
selected_windows = pd.DataFrame(selected_rows)
selected_windows.to_csv(FIGURE_DATA / "selected_windows.csv", index=False)

,Source,Feature,Mean,Standard deviation,5th percentile,Median,95th percentile
0,Observed held-out,Daily log return (%),0.068116,1.563946,-2.505497,0.14331,2.258267
1,Observed held-out,Twenty-day return volatility (%),1.376433,0.766104,0.601944,1.214157,2.506354
2,Generated,Daily log return (%),0.108122,0.066878,0.040576,0.099414,0.224152
3,Generated,Twenty-day return volatility (%),2.50234,0.069372,2.462855,2.468814,2.660486
4,Time-shuffled,Daily log return (%),0.068116,1.563946,-2.505497,0.14331,2.258267
5,Time-shuffled,Twenty-day return volatility (%),1.376433,0.766104,0.601944,1.214157,2.506354


In [4]:
figure, axes = plt.subplots(1, 2, figsize=(11.2, 4.7), constrained_layout=True)
x = np.arange(len(feature_names))
width = 0.24
axes[0].bar(x - width, quality["real_lag_one_correlation"], width=width, color=PALETTE["cyan"], label="Observed held-out")
axes[0].bar(x, quality["synthetic_lag_one_correlation"], width=width, color=PALETTE["green"], label="Generated")
axes[0].bar(x + width, shuffled_quality["synthetic_lag_one_correlation"], width=width, color=PALETTE["rose"], label="Time-shuffled")
axes[0].set_xticks(x, ["Daily return", "Twenty-day volatility"])
axes[0].set_ylabel("Lag-one correlation")
axes[0].set_title("Adjacent-Value Dependence")
axes[0].legend(fontsize=8.5)
diagnostic_labels = ["Prediction error after\ngenerated windows", "Prediction error after\nobserved windows", "Balanced\nclassification accuracy"]
axes[1].bar(np.arange(3) - 0.18, scalar_quality["Generated windows"], width=0.36, color=PALETTE["green"], label="Generated")
axes[1].bar(np.arange(3) + 0.18, scalar_quality["Time-shuffled windows"], width=0.36, color=PALETTE["rose"], label="Time-shuffled")
axes[1].set_xticks(np.arange(3), diagnostic_labels)
axes[1].set_ylabel("Diagnostic value")
axes[1].set_title("Prediction and Separation Checks")
axes[1].legend()
for axis in axes:
    finish_axis(axis)
save_figure(figure, "held_out_diagnostics")
display(figure)
plt.close(figure)

<Figure size 1120x470 with 2 Axes>

In [5]:
figure, axes = plt.subplots(1, 2, figsize=(11.2, 4.6), constrained_layout=True)
source_colors = {"Observed held-out": PALETTE["cyan"], "Generated": PALETTE["green"], "Time-shuffled": PALETTE["rose"]}
for axis, feature in zip(axes, feature_names):
    part = distribution_values[distribution_values["Feature"] == feature]
    for source in source_colors:
        values = part[part["Source"] == source]["Value"]
        axis.hist(values, bins=55, density=True, histtype="step", linewidth=1.8, color=source_colors[source], label=source)
    axis.set_xlabel(feature)
    axis.set_ylabel("Density")
    axis.set_title(feature)
    finish_axis(axis)
axes[0].legend(fontsize=8.5)
save_figure(figure, "distribution_comparison")
display(figure)
plt.close(figure)

<Figure size 1120x460 with 2 Axes>

In [6]:
figure, axes = plt.subplots(2, 1, figsize=(11.2, 7.0), constrained_layout=True)
for axis, feature in zip(axes, feature_names):
    for source, color in [("Observed held-out", PALETTE["cyan"]), ("Generated", PALETTE["green"])]:
        for window_number in range(1, 5):
            part = selected_windows[(selected_windows["Source"] == source) & (selected_windows["Window"] == window_number) & (selected_windows["Feature"] == feature)]
            axis.plot(part["Relative day"], part["Value"], color=color, alpha=0.28 if window_number > 1 else 0.9, linewidth=1.4, label=source if window_number == 1 else None)
    axis.set_ylabel(feature)
    axis.set_title(feature)
    finish_axis(axis)
axes[-1].set_xlabel("Relative day in window")
axes[0].legend()
save_figure(figure, "selected_real_generated_windows")
display(figure)
plt.close(figure)

<Figure size 1120x700 with 2 Axes>

## A Deliberately Broken Comparison

A time-shuffled window preserves its observed values but changes their order:

$$
\widetilde X_{i,t}=X_{i,\pi_i(t)}.
$$

Here, $X_{i,t}$ is the value at position $t$ in window $i$; $\pi_i$ is a random permutation of the positions in that window; $\widetilde X_{i,t}$ is the reordered value. This construction leaves each window's marginal values unchanged while deliberately damaging temporal order. It checks whether the diagnostics respond to an obvious loss of sequence information.

In [7]:
known_generator = np.random.default_rng(891)
constant_sequences = np.repeat(known_generator.normal(size=(100, 1, 1)), 5, axis=1)
independent_sequences = known_generator.normal(size=(100, 5, 1))
known_quality = timegan_quality_report(constant_sequences[:60], constant_sequences[60:], independent_sequences[:60], independent_sequences[60:])
assert known_quality["lag_one_absolute_gap"][0] > 0.7
assert known_quality["synthetic_trained_prediction_mse"] > known_quality["real_trained_prediction_mse"] + 0.1

destroyed_table = pd.DataFrame({
    "Diagnostic": ["Lag-one absolute gap", "Next-step error after fitting independent windows", "Next-step error after fitting constant windows"],
    "Value": [known_quality["lag_one_absolute_gap"][0], known_quality["synthetic_trained_prediction_mse"], known_quality["real_trained_prediction_mse"]],
    "Expected relation": ["Greater than 0.7", "More than 0.1 above the error from constant windows", "Reference value"],
})
destroyed_table.to_csv(FIGURE_DATA / "destroyed_temporal_check.csv", index=False)
save_table(destroyed_table, "destroyed_temporal_check", "Controlled check after temporal information is deliberately destroyed.", "tab:chapter4-destroyed-temporal-check", fit=True)

limits = pd.DataFrame({
    "Diagnostic": ["Lag-one correlation", "Next-step prediction error", "Balanced classification accuracy", "Feature summaries", "Selected windows"],
    "What it checks": ["One-step linear dependence", "Transfer of a simple predictive relation", "Separation by one fitted classifier", "Marginal location and spread", "Visible finite-window behavior"],
    "What it does not establish": ["Long memory or nonlinear dependence", "Trading value or broad forecasting skill", "Equality of the full distributions", "Temporal order", "A coherent unlimited market history"],
})
save_table(limits, "diagnostic_limits", "Interpretation limits for the reported diagnostics.", "tab:chapter4-diagnostic-limits", fit=True)

,Diagnostic,Value,Expected relation
0,Lag-one absolute gap,1.062505,Greater than 0.7
1,Next-step error after fitting independent windows,1.235428,More than 0.1 above the error from constant wi...
2,Next-step error after fitting constant windows,1.51579e-05,Reference value


,Diagnostic,What it checks,What it does not establish
0,Lag-one correlation,One-step linear dependence,Long memory or nonlinear dependence
1,Next-step prediction error,Transfer of a simple predictive relation,Trading value or broad forecasting skill
2,Balanced classification accuracy,Separation by one fitted classifier,Equality of the full distributions
3,Feature summaries,Marginal location and spread,Temporal order
4,Selected windows,Visible finite-window behavior,A coherent unlimited market history


,Diagnostic,What it checks,What it does not establish
0,Lag-one correlation,One-step linear dependence,Long memory or nonlinear dependence
1,Next-step prediction error,Transfer of a simple predictive relation,Trading value or broad forecasting skill
2,Balanced classification accuracy,Separation by one fitted classifier,Equality of the full distributions
3,Feature summaries,Marginal location and spread,Temporal order
4,Selected windows,Visible finite-window behavior,A coherent unlimited market history


## Tests

In [8]:
assert np.all((quality["balanced_discriminator_accuracy"] >= 0, quality["balanced_discriminator_accuracy"] <= 1))
assert quality["synthetic_trained_prediction_mse"] >= 0
assert quality["real_trained_prediction_mse"] >= 0
assert np.isfinite(quality["lag_one_absolute_gap"]).all()
assert np.allclose(model.sample(12, window_length, seed=4200).numpy(), model.sample(12, window_length, seed=4200).numpy(), atol=0)
assert not np.allclose(model.sample(12, window_length, seed=4200).numpy(), model.sample(12, window_length, seed=4201).numpy())

diagnostic_checks = pd.DataFrame({
    "Check": ["Fresh generated training windows", "Fresh generated held-out windows", "Finite lag-one gaps", "Nonnegative prediction errors", "Balanced accuracy in unit interval", "Equal seeds reproduce samples", "Different seeds change samples", "Destroyed temporal information detected"],
    "Result": ["Pass", "Pass", "Pass", "Pass", "Pass", "Pass", "Pass", "Pass"],
})
save_table(diagnostic_checks, "diagnostic_checks", "Checks for fresh samples and held-out diagnostic behavior.", "tab:chapter4-diagnostic-checks", fit=True)

,Check,Result
0,Fresh generated training windows,Pass
1,Fresh generated held-out windows,Pass
2,Finite lag-one gaps,Pass
3,Nonnegative prediction errors,Pass
4,Balanced accuracy in unit interval,Pass
5,Equal seeds reproduce samples,Pass
6,Different seeds change samples,Pass
7,Destroyed temporal information detected,Pass


,Check,Result
0,Fresh generated training windows,Pass
1,Fresh generated held-out windows,Pass
2,Finite lag-one gaps,Pass
3,Nonnegative prediction errors,Pass
4,Balanced accuracy in unit interval,Pass
5,Equal seeds reproduce samples,Pass
6,Different seeds change samples,Pass
7,Destroyed temporal information detected,Pass
